# M3 baselines vs human labels

Compare primary human labels (`human_annotation`) with Milestone 3 baselines on each row:

- **Phase 1:** `baseline_scores.m3.lexical.sequence_ratio`
- **Phase 2 (optional):** `m3.bertscore.f1` if you ran with `--bertscore`
- **Phase 3 (optional):** `m3.nli.label` if you ran with `--nli`

**Prerequisite:** run from the repository root (or adjust `ROOT` below) and use `pip install -e '.[dev]'` or `PYTHONPATH=.` so `rde_eval` imports resolve.

Rebuild `results/samples_with_m3.jsonl` after changing phases, for example:

```bash
python scripts/run_baselines.py --input data/samples.jsonl --output results/samples_with_m3.jsonl
python scripts/run_baselines.py --input data/samples.jsonl --output results/samples_with_m3.jsonl --bertscore
python scripts/run_baselines.py --input data/samples.jsonl --output results/samples_with_m3.jsonl --nli --nli-model facebook/roberta-large-mnli
```

In [ ]:
from __future__ import annotations

import json
from collections import defaultdict
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").exists() and (ROOT.parent / "pyproject.toml").exists():
    ROOT = ROOT.parent

PATH = ROOT / "results" / "samples_with_m3.jsonl"
if not PATH.is_file():
    PATH = ROOT / "data" / "samples.jsonl"
    print("Using data/samples.jsonl — run run_baselines.py to create results/samples_with_m3.jsonl")

print("ROOT =", ROOT)
print("JSONL =", PATH)

In [ ]:
def load_jsonl(path: Path) -> list[dict]:
    rows: list[dict] = []
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


rows = load_jsonl(PATH)
len(rows)

In [ ]:
from rde_eval.baselines import merge_milestone3_baseline

prepared: list[dict] = []
for r in rows:
    r = dict(r)
    if "baseline_scores" not in r or "m3" not in (r.get("baseline_scores") or {}):
        r["baseline_scores"] = merge_milestone3_baseline(
            r.get("baseline_scores"), source=str(r["source"]), output=str(r["output"])
        )
    prepared.append(r)

rows = prepared

## Mean lexical ratio by human primary label

Exploratory: higher overlap does not imply RDE "Preserved" — this is only a string-similarity proxy.

In [ ]:
by_label: dict[str, list[float]] = defaultdict(list)
for r in rows:
    label = r.get("human_annotation")
    if not label:
        continue
    ratio = r["baseline_scores"]["m3"]["lexical"]["sequence_ratio"]
    by_label[str(label)].append(ratio)

for label in sorted(by_label):
    vals = by_label[label]
    mean = sum(vals) / len(vals)
    print(f"{label:22}  mean_ratio={mean:.4f}  n={len(vals)}")

## Optional: BERTScore F1 and NLI label by `human_annotation`

If the JSONL was produced with `--bertscore` or `--nli`, the next cell prints simple aggregates. Otherwise it prints a short hint.

In [ ]:
from collections import Counter, defaultdict

by_h_f1: dict[str, list[float]] = defaultdict(list)
by_h_nli: dict[str, list[str]] = defaultdict(list)

for r in rows:
    ha = r.get("human_annotation")
    if not ha:
        continue
    ha = str(ha)
    m3 = (r.get("baseline_scores") or {}).get("m3") or {}
    bs = m3.get("bertscore") or {}
    if isinstance(bs, dict) and "f1" in bs:
        by_h_f1[ha].append(float(bs["f1"]))
    nl = m3.get("nli") or {}
    if isinstance(nl, dict) and nl.get("label"):
        by_h_nli[ha].append(str(nl["label"]))

if by_h_f1:
    print("BERTScore F1 (mean) by human_annotation")
    for lbl in sorted(by_h_f1):
        vals = by_h_f1[lbl]
        print(f"{lbl:22}  mean_f1={sum(vals) / len(vals):.4f}  n={len(vals)}")
else:
    print("No m3.bertscore in rows — rerun run_baselines with --bertscore if you want this block.")

print()
if by_h_nli:
    print("NLI predicted label counts by human_annotation (top 5 labels each)")
    for lbl in sorted(by_h_nli):
        ctr = Counter(by_h_nli[lbl])
        top = ", ".join(f"{k}:{v}" for k, v in ctr.most_common(5))
        print(f"{lbl:22}  {top}")
else:
    print("No m3.nli in rows — rerun run_baselines with --nli if you want this block.")